# Agent offline evaluation: full batch evaluate and portal logging

Runs a full batch evaluation with all quality evaluators against the `aria-rm-briefing-agent` responses and logs results to the Azure AI Foundry portal. After running, click the `studio_url` to view the evaluation run in the Foundry portal.

In [1]:
import hashlib
import json
import os
import subprocess
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    CoherenceEvaluator,
    FluencyEvaluator,
    RelevanceEvaluator,
    GroundednessEvaluator,
    SimilarityEvaluator,
    AzureOpenAIModelConfiguration,
    evaluate,
)
from dotenv import load_dotenv

## Environment

In [2]:
repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ['CHAT_MODEL']

SUB_ID = subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'
AOAI_ENDPOINT    = f'https://aif-core-{SUFFIX}.services.ai.azure.com/'

lab_dir        = repo_root / '08-agents' / '08-06-agent-offline-evaluation'
test_data_path = lab_dir / 'test_data.jsonl'
output_path    = lab_dir / 'eval_results.jsonl'

print(f'Project endpoint : {PROJECT_ENDPOINT}')

Project endpoint : https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f


## Model configuration

In [3]:
credential = DefaultAzureCredential()

# Python 3.13 + azure-ai-evaluation 1.16.x workaround: omit `credential` from
# AzureOpenAIModelConfiguration (SDK validates via isinstance(value, Any), which
# Python 3.13 made into a hard TypeError). Pass credential as a kwarg to each
# evaluator below instead. See 08-06-00 README for details.
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AOAI_ENDPOINT,
    azure_deployment=CHAT_MODEL,
)


## azure_ai_project: enables portal logging

Passing `azure_ai_project` to `evaluate()` causes the SDK to upload results to Foundry and return a `studio_url`.

In [4]:
azure_ai_project = PROJECT_ENDPOINT

## Full batch evaluate()

In [5]:
results = evaluate(
    data=str(test_data_path),
    evaluators={
        'coherence':    CoherenceEvaluator(model_config=model_config, credential=credential),
        'fluency':      FluencyEvaluator(model_config=model_config, credential=credential),
        'relevance':    RelevanceEvaluator(model_config=model_config, credential=credential),
        'groundedness': GroundednessEvaluator(model_config=model_config, credential=credential),
        'similarity':   SimilarityEvaluator(model_config=model_config, credential=credential),
    },
    evaluator_config={
        'coherence':    {'column_mapping': {'query': '${data.query}', 'response': '${data.response}'}},
        'fluency':      {'column_mapping': {'query': '${data.query}', 'response': '${data.response}'}},
        'relevance':    {'column_mapping': {'query': '${data.query}', 'response': '${data.response}', 'context': '${data.context}'}},
        'groundedness': {'column_mapping': {'query': '${data.query}', 'response': '${data.response}', 'context': '${data.context}'}},
        'similarity':   {'column_mapping': {'query': '${data.query}', 'response': '${data.response}', 'ground_truth': '${data.ground_truth}'}},
    },
    azure_ai_project=azure_ai_project,
    output_path=str(output_path),
)

2026-05-11 12:50:19 +0200 134590891411136 azure.ai.evaluation._legacy.prompty._prompty WARNING  [0/10] AsyncAzureOpenAI request failed. RateLimitError: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}. Retrying in 30.000000 seconds.
Traceback (most recent call last):
  File "<repo-root>/.venv/lib/python3.13/site-packages/azure/ai/evaluation/_legacy/prompty/_prompty.py", line 383, in _send_with_retries
    response = await client.chat.completions.create(**params)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<repo-root>/.venv/lib/python3.13/site-packages/azure/ai/evaluation/_legacy/_batch_engine/_openai_injector.py", line 50, in async_wrapper
    result: _WithUsage = await method(*args, **kwargs)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<repo-root>/.venv/lib/python3.13/site-packages/openai/resources/chat/completions/completions.py", line 2714, in create
   

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "coherence_20260511_105016_232274"
Run status: "Completed"
Start time: "2026-05-11 10:50:16.232274+00:00"
Duration: "0:00:08.298698"

2026-05-11 12:50:51 +0200 134590891411136 execution.bulk     INFO     Finished 4 / 5 lines.
2026-05-11 12:50:51 +0200 134590891411136 execution.bulk     INFO     Average execution time for completed lines: 8.79 seconds. Estimated time for incomplete lines: 8.79 seconds.
2026-05-11 12:50:51 +0200 134590891411136 execution.bulk     INFO     Finished 5 / 5 lines.
2026-05-11 12:50:51 +0200 134590891411136 execution.bulk     INFO     Average execution time for completed lines: 7.11 seconds. Estimated time for incomplete lines: 0.0 seconds.
======= Run Summary =======

Run name: "relevance_20260511_105016_236925"
Run status: "Completed"
Start time: "2026-05-11 10:50:16.236925+00:00"
Duration: "0:00:35.680776"

2026-05-11 12:50:52 +0200 134590899803840 execution.bulk     INFO     Finished 5 / 5 lines.
2026-05-11 12:50:52 +

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "fluency_20260511_105016_235264"
Run status: "Completed"
Start time: "2026-05-11 10:50:16.235264+00:00"
Duration: "0:00:36.303476"

2026-05-11 12:50:52 +0200 134590883018432 execution.bulk     INFO     Finished 5 / 5 lines.
2026-05-11 12:50:52 +0200 134590883018432 execution.bulk     INFO     Average execution time for completed lines: 7.33 seconds. Estimated time for incomplete lines: 0.0 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "groundedness_20260511_105016_239404"
Run status: "Completed"
Start time: "2026-05-11 10:50:16.239404+00:00"
Duration: "0:00:37.417738"

======= Combined Run Summary (Per Evaluator) =======

{
    "coherence": {
        "status": "Completed",
        "duration": "0:00:08.298698",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "fluency": {
        "status": "Completed",
        "duration": "0:00:36.303476",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "relevance": {
        "status": "Completed",
        "duration": "0:00:35.680776",
        "completed_lines": 5,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
   

View evaluation results in Foundry portal:
https://ai.azure.com/resource/build/evaluation/3e864d4f-d92d-4182-a49f-f1118878762c?wsid=/subscriptions/00000000-0000-0000-0000-000000000000/resourceGroups/rg-foundry-core-c2676f/providers/Microsoft.CognitiveServices/accounts/aif-core-c2676f/projects/project-admin-c2676f&tid=b845d325-6786-435a-bc28-b326d9fcbe16


<repo-root>/08-agents/08-06-agent-offline-evaluation/evaluation_helpers.py:107: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_df = df.style.applymap(highlight_scores, subset=existing_score_cols).hide(axis="index")


## Studio URL: view results in Foundry portal

In [6]:
studio_url = results.get('studio_url')
if studio_url:
    print(f'View evaluation results in Foundry portal:\n{studio_url}')
else:
    print('No studio_url returned. Check that azure_ai_project is correct.')
    print('Metrics:', results.get('metrics', {}))

## Metrics summary

In [7]:
import sys
sys.path.insert(0, str(lab_dir))
from evaluation_helpers import display_metrics_summary, display_row_results

display_metrics_summary(results.get('metrics', {}))

### Aggregate Metrics Summary

#### Quality Metrics

Metric,Score
coherence.coherence,4.20
coherence.gpt_coherence,4.20
fluency.fluency,4.00
fluency.gpt_fluency,4.00
relevance.relevance,5.00
relevance.gpt_relevance,5.00
coherence.binary_aggregate,1.00
fluency.binary_aggregate,1.00
relevance.binary_aggregate,1.00


#### RAG & Similarity Metrics

Metric,Score
groundedness.groundedness,5.00
groundedness.gpt_groundedness,5.00
similarity.similarity,3.60
similarity.gpt_similarity,3.60
groundedness.binary_aggregate,1.00
similarity.binary_aggregate,0.80


## Row-level results

In [8]:
display_row_results(
    results.get('rows', []),
    columns=['coherence', 'fluency', 'relevance', 'groundedness', 'similarity'],
)

### Row-Level Results

#,Query,Coherence,Fluency,Relevance,Groundedness,Similarity
1,I have a 9am with the Berger family for ...,5.000000,4.000000,5.000000,5.000000,4.000000
2,Show me the Lindemann family office port...,4.000000,4.000000,5.000000,5.000000,4.000000
3,Anything been written recently about AI ...,4.000000,4.000000,5.000000,5.000000,4.000000
4,Summarise what is been happening on the ...,4.000000,4.000000,5.000000,5.000000,5.000000
5,Get me the FINMA sustainability disclosu...,4.000000,4.000000,5.000000,5.000000,1.000000
